## 这份代码在做什么  
你要预测加州房价（回归问题） 

Baseline：只用原始特征（例如收入、房间数、纬度经度…）训练 ANN  

Enhanced：在原始特征基础上，额外加一列**“邻居平均房价”**（用地理位置找附近的房子，把这些邻居的训练房价取平均），再训练 ANN  

最后比较两者的 Test MSE，看加邻域特征有没有让模型更准。


## method  
我们的方法在 baseline ANN 的基础上加入了一个基于 “地理邻域” 的特征来显式建模空间相关性。   

虽然原始特征中已包含纬度与经度，但仅将它们作为普通数值输入会要求模型自行学习复杂的“位置—房价”非线性关系。例如靠海房价贵，经纬度接近的房子房价也接近等。  

为此，我们使用纬度/经度在空间中为每个样本定义k-近邻，并用训练集中这些邻居的目标值（房价中位数）计算邻域平均，得到新特征 Avg_Neighbor_Price。 
 
该特征作为额外输入与原始特征一同输入到同结构的 ANN 中，从而为模型提供“附近地区价格相近”的先验信息，提升对位置效应的表达能力，同时在测试集上构造该特征时仅使用训练集标签以避免数据泄露。

## 1. 导入库
numpy (np)：数学计算用的，尤其是 mean、数组索引等  
pandas (pd)：表格数据工具（DataFrame），像 Excel 那种结构  
matplotlib.pyplot (plt)：画图用（散点图、直线等）  

scikit-learn 这些：  
fetch_california_housing：下载/读取 California Housing 数据集（已经内置）  
train_test_split：把数据随机分成训练集/测试集  
StandardScaler：做标准化（让特征均值=0，方差=1），神经网络很需要这个  
MLPRegressor：前馈神经网络（ANN）做回归  
mean_squared_error：算 MSE（均方误差）  
NearestNeighbors：KNN 找最近的邻居（你用经纬度找附近地区）  

In [ ]:
# To make sure everyone has these necessary modules
%pip install numpy pandas matplotlib scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [67]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import NearestNeighbors

from sklearn.ensemble import GradientBoostingRegressor


## 1.1 绘图工具函数（后面每个模型训练完直接出图）
为了让 report 更清晰，我们在每个模型后都画三张图：
- Pred vs True（拟合散点）
- Residual histogram（残差分布）
- Residual vs Pred（残差随预测值变化）

In [ ]:
# Plot helper functions (used after each model)
import numpy as np
import matplotlib.pyplot as plt

def plot_pred_vs_true(y_true, y_pred, title, alpha=0.35):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    plt.figure(figsize=(6,6))
    plt.scatter(y_true, y_pred, alpha=alpha)
    mn = min(y_true.min(), y_pred.min())
    mx = max(y_true.max(), y_pred.max())
    plt.plot([mn, mx], [mn, mx])
    plt.xlabel("True values")
    plt.ylabel("Predicted values")
    plt.title(title)
    plt.tight_layout()
    plt.show()

def plot_residual_hist(y_true, y_pred, title, bins=30):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    resid = y_true - y_pred
    plt.figure(figsize=(6,4))
    plt.hist(resid, bins=bins)
    plt.xlabel("Residual (true - pred)")
    plt.ylabel("Count")
    plt.title(title)
    plt.tight_layout()
    plt.show()

def plot_residual_vs_pred(y_true, y_pred, title, alpha=0.35):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    resid = y_true - y_pred
    plt.figure(figsize=(6,4))
    plt.scatter(y_pred, resid, alpha=alpha)
    plt.axhline(0)
    plt.xlabel("Predicted values")
    plt.ylabel("Residual (true - pred)")
    plt.title(title)
    plt.tight_layout()
    plt.show()


## 2. 读取数据：X是特征，y是答案  
X：输入特征（每一行是一块区域的统计信息），比如：  
	•	MedInc（中位收入）  
	•	HouseAge（房龄）  
	•	AveRooms（平均房间数）  
	•	Latitude / Longitude（纬度经度）等等  

y：目标值（你要预测的真实房价标签），一个一维数组。


In [53]:
# Load data
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target  # median house value

print("Original X (head):")
print(X.head())

Original X (head):
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  


## 3. 切分训练集/测试集。
test_size=0.2：20% 做测试集，80% 做训练集  
random_state=42：固定随机种子，保证你每次跑切分结果都一样（方便写报告、复现）。

#### 为什么要切分（split）  
因为你后面要做“邻居平均房价”这个特征，如果你不先 split，很容易把测试集的真实房价 y_test 偷偷用进来，这叫 data leakage（数据泄露），会让结果看起来特别好但其实作弊了。  

你现在的写法是正确的：  
✅ 先 split  
✅ 用训练集构造特征  
✅ 测试集只能“借用训练集的信息”，不能用测试集真实标签

In [54]:
# Train/test split (avoid leakage)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 4.核心： 邻域特征add_graph_features  
#### 4.1 直觉理解  
房价强烈跟地理位置相关：  
你住伦敦一区，隔壁一区差不多也贵；你住郊区，隔壁郊区也差不多便宜。  
所以我们给每一条数据加一个新特征：  
Avg_Neighbor_Price = “附近 k 个邻居的平均房价”  
这样神经网络不需要自己从经纬度“猜”空间规律，我们直接把空间规律喂给它。  

#### 4.2 函数代码讲解  
见代码  

#### * 为什么要k+1  
当你给训练集加特征时：df 和 reference_df 是同一个集合。  
那 KNN 找最近邻时，第一个邻居往往就是它自己（距离=0）。  
如果你不排除它自己，那么：  
Avg_Neighbor_Price = mean( [自己的y, 邻居1的y, 邻居2的y, …] )  
这会让新特征更像“答案本身”，会让 enhanced 的效果“看起来更强”（偏虚高）。  
所以我们做：  
	•	先找 k+1 个邻居  
	•	把第一个（自己）丢掉  
	•	剩下 k 个再平均  



In [55]:
def add_graph_features(df, reference_df, reference_y, k=5, exclude_self=False):
    """Add a simple neighbourhood feature based on geographic kNN.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset to which we want to add the feature (train or test).
    reference_df : pd.DataFrame
        The pool of candidate neighbours (typically the training set only).
    reference_y : array-like
        Target values aligned with reference_df (must be y_train to avoid leakage).
    k : int, default=5
        Number of neighbours used to compute the neighbourhood statistic.
    exclude_self : bool, default=False
        If True (typically for training set when df == reference_df), drop the closest
        neighbour which is usually the point itself.

    Returns
    -------
    pd.DataFrame or None
        A copy of df with an added column 'Avg_Neighbor_Price'. Returns None for invalid input.
    """
    # Defensive checks (match assignment-style behaviour used elsewhere in this notebook)
    if df is None or reference_df is None or reference_y is None:
        return None
    if len(df) == 0:
        out = df.copy()
        out['Avg_Neighbor_Price'] = np.array([])
        return out

    # 1) Coordinates for neighbour search (lat/long)
    coords_ref = reference_df[['Latitude', 'Longitude']].to_numpy()
    coords_target = df[['Latitude', 'Longitude']].to_numpy()

    # 2) If excluding self, grab one extra neighbour then drop the first one
    n_neighbors = k + 1 if exclude_self else k

    # 3) Fit neighbour index once, then query all targets
    nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(coords_ref)
    _distances, indices = nbrs.kneighbors(coords_target)

    # 4) Vectorised average neighbour price (faster + cleaner than Python loops)
    if exclude_self:
        indices = indices[:, 1:]
    ref_y = np.asarray(reference_y)
    avg_neighbor_price = ref_y[indices].mean(axis=1)

    out = df.copy()
    out['Avg_Neighbor_Price'] = avg_neighbor_price
    return out


#### 4.3 用add_graph_features函数生成enhanced数据  
关键点：  
训练集 enhanced：邻居来自训练集本身，但排除自己（exclude_self=True）  
测试集 enhanced：邻居只能来自训练集（reference_df = X_train_raw, reference_y = y_train）  
✅ 不使用 y_test（避免泄露）

In [56]:
# Enhanced datasets
# - training enhanced excludes self
X_train_enhanced = add_graph_features(
    X_train_raw, X_train_raw, y_train, k=5, exclude_self=True
)
# - test enhanced uses ONLY training as reference (no leakage)
X_test_enhanced = add_graph_features(
    X_test_raw, X_train_raw, y_train, k=5, exclude_self=False
)

print("\nEnhanced X_train (head):")
print(X_train_enhanced.head())


Enhanced X_train (head):
       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
14196  3.2596      33.0  5.017657   1.006421      2300.0  3.691814     32.71   
8267   3.8125      49.0  4.473545   1.041005      1314.0  1.738095     33.77   
17445  4.1563       4.0  5.645833   0.985119       915.0  2.723214     34.66   
14265  1.9425      36.0  4.002817   1.033803      1418.0  3.994366     32.69   
2271   3.5542      43.0  6.268421   1.134211       874.0  2.300000     36.78   

       Longitude  Avg_Neighbor_Price  
14196    -117.03              1.1614  
8267     -118.16              3.3092  
17445    -120.48              1.4200  
14265    -117.11              0.9806  
2271     -119.80              0.8496  


## 5. 定义神经网络：为什么抽成 build_model  
这样做的意义：  
Baseline 和 Enhanced 用一模一样的网络结构，差异只来自“有没有邻域特征”。
这叫 fair comparison（公平比较）。 

参数解释：  
	•	hidden_layer_sizes=(64,32)：两层隐藏层  
	•	第一层 64 个神经元  
	•	第二层 32 个神经元  
	•	activation='relu'：激活函数 ReLU（非线性，神经网络的灵魂）  
	•	solver='adam'：优化器（训练时怎么更新参数）  
	•	alpha=1e-4：L2 正则化强度（防止过拟合的一点点约束）  
	•	max_iter=500：最多训练 500 次迭代  
	•	random_state=42：固定初始化随机性



In [57]:
#Model definition helper (same architecture for fair comparison)
def build_model(random_state=42):
    return MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        alpha=1e-4,
        max_iter=500,
        random_state=random_state
    )

In [ ]:
def fit_predict_mse(model, X_train, y_train, X_test, y_test, scale=True, random_state=42):
    """Fit a regression model and return (mse, y_pred, fitted_scaler).

    Notes
    -----
    - For MLPRegressor we scale features (important).
    - For tree-based models we typically do NOT need scaling.
    """
    scaler = None
    if scale:
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)
    else:
        Xtr, Xte = X_train, X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    mse = mean_squared_error(y_test, y_pred)
    return mse, y_pred, scaler


## 6.Baseline：不加邻域特征，直接训练 ANN  

#### 6.1 标准化（重要）
为什么要标准化？  
神经网络训练是靠梯度下降，如果某个特征数值范围很大（比如收入 10100），另一个很小（比如经纬度 3040），训练会很不稳定、收敛慢、甚至效果差。  

关键点：  
	•	fit_transform 只对训练集做（学习均值方差并转换）  
	•	测试集只能 transform（用训练集的均值方差来变换）  
	•	✅ 这也是避免泄露（不能用测试集统计量）


In [58]:
# BASELINE: train ANN on original X (no graph features)
scaler_base = StandardScaler()
X_train_base_scaled = scaler_base.fit_transform(X_train_raw)
X_test_base_scaled = scaler_base.transform(X_test_raw)

#### 6.2 训练 + 预测 + MSE  
•fit：训练模型（学参数）  
•predict：预测测试集房价  
•MSE：衡量误差大小  
•越小越好  
•MSE 是平均平方误差：错误大的会被平方惩罚更重   


In [59]:
model_base = build_model(random_state=42)
model_base.fit(X_train_base_scaled, y_train)

y_pred_base = model_base.predict(X_test_base_scaled)
mse_base = mean_squared_error(y_test, y_pred_base)

In [ ]:
# Plots for ANN Baseline
plot_pred_vs_true(y_test, y_pred_base, "ANN Baseline: Pred vs True")
plot_residual_hist(y_test, y_pred_base, "ANN Baseline: Residual histogram")
plot_residual_vs_pred(y_test, y_pred_base, "ANN Baseline: Residual vs Pred")

## 7. Enhanced：加了 Avg_Neighbor_Price 再训练 ANN  
这一段和 baseline 完全同样流程，只是输入特征变成 X_train_enhanced / X_test_enhanced  

这里为什么要单独一个 scaler_enh？  
因为 enhanced 多了一列特征，维度不一样；不能用 baseline 的 scaler。

In [60]:
scaler_enh = StandardScaler()
X_train_enh_scaled = scaler_enh.fit_transform(X_train_enhanced)
X_test_enh_scaled = scaler_enh.transform(X_test_enhanced)

model_enh = build_model(random_state=42)
model_enh.fit(X_train_enh_scaled, y_train)

y_pred_enh = model_enh.predict(X_test_enh_scaled)
mse_enh = mean_squared_error(y_test, y_pred_enh)

In [ ]:
# Plots for ANN Enhanced (+Avg_Neighbor_Price)
plot_pred_vs_true(y_test, y_pred_enh, "ANN Enhanced (+Avg_Neighbor_Price): Pred vs True")
plot_residual_hist(y_test, y_pred_enh, "ANN Enhanced (+Avg_Neighbor_Price): Residual histogram")
plot_residual_vs_pred(y_test, y_pred_enh, "ANN Enhanced (+Avg_Neighbor_Price): Residual vs Pred")

## 8. 轻量可解释性：Permutation test（Avg_Neighbor_Price 的贡献）
> 目标：验证增强模型的提升是否真的来自 `Avg_Neighbor_Price`，而不是偶然噪声。

In [ ]:
# Permutation importance for Avg_Neighbor_Price on the trained Enhanced ANN (main split)
# (uses scaler_enh and model_enh from Section 7)
X_test_perm = X_test_enhanced.copy()
X_test_perm['Avg_Neighbor_Price'] = np.random.RandomState(42).permutation(np.asarray(X_test_perm['Avg_Neighbor_Price']))

X_test_perm_scaled = scaler_enh.transform(X_test_perm)
y_pred_perm = model_enh.predict(X_test_perm_scaled)
mse_perm = mean_squared_error(y_test, y_pred_perm)

delta_mse = mse_perm - mse_enh

print(f"Enhanced ANN MSE (original):   {mse_enh:.6f}")
print(f"Enhanced ANN MSE (permuted):   {mse_perm:.6f}")
print(f"ΔMSE after permuting Avg_Neighbor_Price: {delta_mse:.6f}")

import matplotlib.pyplot as plt
plt.figure(figsize=(5,4))
plt.bar(['ΔMSE'], [delta_mse])
plt.ylabel('Increase in MSE')
plt.title('Permutation impact (Avg_Neighbor_Price)')
plt.tight_layout()
plt.show()


In [ ]:
from IPython.display import display, Markdown

display(Markdown(f"""### Permutation Test 结果解释（轻量可解释性）

我们对增强模型中的 **Avg_Neighbor_Price** 做了 *permutation test*：只把该特征在测试集里随机打乱，其它特征不变，然后重新计算误差。

- 原始 Enhanced ANN 的 MSE：**{mse_enh:.3f}**
- 打乱 Avg_Neighbor_Price 后的 MSE：**{mse_perm:.3f}**
- MSE 增量（ΔMSE）：**{delta_mse:.3f}**

**结论：** MSE 明显上升说明模型确实“在用”这个邻域价格特征，它不是噪声；打乱后局部空间信息被破坏，预测会变差。

**为什么加入 Avg_Neighbor_Price 会让模型更好/更容易学？（可能原因）**
1. **空间自相关（spatial autocorrelation）**：房价在地理空间上通常高度相关，临近区域往往共享配套、学区、交通与环境等因素，价格自然相近。
2. **把“难学的经纬度关系”变成“可直接用的信号”**：如果只给 Latitude/Longitude，模型需要自己学到复杂的空间映射（nonlinear mapping）才能间接推断“附近贵不贵”；而 Avg_Neighbor_Price 相当于直接提供了一个 *local market summary*，降低学习难度，所以更容易拟合、泛化更好。
3. **补充隐含变量**：数据里未显式给出的 neighbourhood quality、政策、供需等，往往被邻居成交价/房价所“吸收”，因此该特征能作为这些隐含因素的代理变量（proxy）。

因此，Permutation test 的结果为我们提供了一个轻量但有效的证据：**Enhanced 的提升主要来自邻域价格信息的贡献**。"""))

## 9. 对比结果：MSE 和提升百分比  
如果 enhanced 更好：mse_enh 更小 → improvement 是正数（提升%）  
如果 enhanced 更差：improvement 是负数（退步%）


## 10. 拓展模型：树模型（Gradient Boosting） + 邻域特征  
前面我们用的是 ANN（MLPRegressor）。这里再加一个 **非神经网络** 的回归模型做对照：  
- 树模型对特征缩放不敏感（一般不需要 StandardScaler）  
- 能捕捉非线性与特征交互  
- 可以验证：邻域特征提升是否“只对 ANN 有效”，还是对别的模型也有用  


In [ ]:
# Gradient Boosting Regressor (tree-based): no scaling required

# Baseline GBRT (raw features)
gbr_base = GradientBoostingRegressor(random_state=42)
mse_gbr_base, y_pred_gbr_base, _ = fit_predict_mse(gbr_base, X_train_raw, y_train, X_test_raw, y_test, scale=False)
print(f"GBRT Baseline (raw X) Test MSE = {mse_gbr_base:.4f}")

# Enhanced GBRT (add Avg_Neighbor_Price; use a sensible local k, e.g. k=5)
k_ext = 5
X_train_en_ext = add_graph_features(X_train_raw, X_train_raw, y_train, k=k_ext, exclude_self=True)
X_test_en_ext  = add_graph_features(X_test_raw,  X_train_raw, y_train, k=k_ext, exclude_self=False)

gbr_en = GradientBoostingRegressor(random_state=42)
mse_gbr_en, y_pred_gbr_en, _ = fit_predict_mse(gbr_en, X_train_en_ext, y_train, X_test_en_ext, y_test, scale=False)
print(f"GBRT Enhanced (k={k_ext}) Test MSE = {mse_gbr_en:.4f}")

improvement_pct = (mse_gbr_base - mse_gbr_en) / mse_gbr_base * 100
print(f"GBRT improvement from neighbour feature: {improvement_pct:.2f}%")


In [ ]:
# Plots for GBRT Baseline / Enhanced (immediately after the model)
plot_pred_vs_true(y_test, y_pred_gbr_base, "GBRT Baseline: Pred vs True")
plot_residual_hist(y_test, y_pred_gbr_base, "GBRT Baseline: Residual histogram")
plot_residual_vs_pred(y_test, y_pred_gbr_base, "GBRT Baseline: Residual vs Pred")

plot_pred_vs_true(y_test, y_pred_gbr_en, f"GBRT Enhanced (+Avg_Neighbor_Price, k={k_ext}): Pred vs True")
plot_residual_hist(y_test, y_pred_gbr_en, f"GBRT Enhanced (+Avg_Neighbor_Price, k={k_ext}): Residual histogram")
plot_residual_vs_pred(y_test, y_pred_gbr_en, f"GBRT Enhanced (+Avg_Neighbor_Price, k={k_ext}): Residual vs Pred")
